# Data Preprocessing

This notebook is the base for :
- Exploratory Data Analysis
- Data cleaning
- Data Impuation
- Feature Engineering

## Used libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from scipy.stats import skew, kurtosis, binom
from sklearn.linear_model import LinearRegression

## Loading data

In [12]:
x_train = pd.read_csv('../x_train.csv', index_col='ID')
y_train = pd.read_csv('../y_train.csv', index_col='ID')
train = pd.concat([x_train, y_train], axis=1)
test = pd.read_csv('../x_test.csv', index_col='ID')
train.head()

,DATE,STOCK,INDUSTRY,INDUSTRY_GROUP,SECTOR,SUB_INDUSTRY,RET_1,VOLUME_1,RET_2,VOLUME_2,...,VOLUME_16,RET_17,VOLUME_17,RET_18,VOLUME_18,RET_19,VOLUME_19,RET_20,VOLUME_20,RET
ID,,,,,,,,,,,,,,,,,,,,,
0,0,2,18,5,3,44,-0.015748,0.147931,-0.015504,0.179183,...,0.630899,0.003254,-0.379412,0.008752,-0.110597,-0.012959,0.174521,-0.002155,-0.000937,True
1,0,3,43,15,6,104,0.003984,NaN,-0.090580,NaN,...,NaN,0.003774,NaN,-0.018518,NaN,-0.028777,NaN,-0.034722,NaN,True
2,0,4,57,20,8,142,0.000440,-0.096282,-0.058896,0.084771,...,-0.010336,-0.017612,-0.354333,-0.006562,-0.519391,-0.012101,-0.356157,-0.006867,-0.308868,False
3,0,8,1,1,1,2,0.031298,-0.429540,0.007756,-0.089919,...,0.012105,0.033824,-0.290178,-0.001468,-0.663834,-0.013520,-0.562126,-0.036745,-0.631458,False
4,0,14,36,12,5,92,0.027273,-0.847155,-0.039302,-0.943033,...,-0.277083,-0.012659,0.139086,0.004237,-0.017547,0.004256,0.579510,-0.040817,0.802806,False


The train and test inputs are composed of 46 features.

The target of this challenge is `RET` and corresponds to the fact that the **return is in the top 50% of highest stock returns**.

Since the median is very close to 0, this information should not change much with the idea to predict the sign of the return.

## Exploratory Data Analysis

In [ ]:
print(f"The train dataset contains {x_train.shape[0]} rows and {x_train.shape[1]} columns.")
print(f"The test dataset contains {test.shape[0]} rows and {test.shape[1]} columns.")
print(f'Features in the dataset: {x_train.columns}')

The dataset is made of 46 descriptive features: (all float / int values)

- `DATE`: an index of the date (the dates are randomized and anonymized so there is no continuity or link between any dates),
- `STOCK`: an index of the stock,
- `INDUSTRY`: an index of the stock industry domain (e.g., aeronautic, IT, oil company),
- `INDUSTRY_GROUP`: an index of the group industry,
- `SUB_INDUSTRY`: a lower level index of the industry,
- `SECTOR`: an index of the work sector,
- `RET_1` to `RET_20`: the historical residual returns among the last 20 days (i.e., `RET_1` is the return of the previous day and so on),
- `VOLUME_1` to `VOLUME_20`: the historical relative volume traded among the last 20 days (i.e., `VOLUME_1` is the relative volume of the previous day and so on),
The target variable: (binary)

- `RET`: the sign of the residual stock return at time t

Let's have a look on the structure of the data. In particular, I look at missing values, the balance of the dataset and potential correlation between features of the dataset.

In [ ]:
# Plotting missing values per features
missing_values = x_train.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)
missing_values.plot(kind='bar', figsize=(15, 5), title='Missing values per feature')


In [ ]:
# Plotting the distribution of the missing values per category
plt.figure(figsize=(15,10))
plt.subplots_adjust(hspace=0.5)
for i,category in enumerate(['INDUSTRY', 'INDUSTRY_GROUP', 'SECTOR', 'SUB_INDUSTRY', 'STOCK', 'DATE']): 
    plt.subplot(3,2,i+1)
    plt.title(category)
    plt.bar(train[category].sort_values().unique(),
            [(train[train[category]==sub_category].isna().sum(axis=1)>0).sum()/len(train[train[category]==sub_category])*100 for sub_category in train[category].sort_values().unique()])
    plt.xlabel('sub-category')
    plt.ylabel('%')
plt.show()

The plots show the distribution of NaN values per different type of category in percentage. We observe a "relatively" even distribution of NaN values for the categorical variable ``SECTOR``. However, the amount of NaN values for the other categorical variables appears to be less evenly distributed. It might be worthwhile to investigate if there are rows that predominantly consist of NaN values for the descriptive variables ``RET`` and ``VOLUME``. If such rows exist, we can drop them in good faith since these columns do not contribute to understanding the underlying structure. During this investigation, I noticed the following:

Given no observed returns, there is no volume observed. Therefore, we should only delete those observations where no return has been observed over the past days.

In [13]:

ret_cols = [col for col in train.columns if 'RET_' in col]
volume_cols = [col for col in train.columns if 'VOLUME_' in col]

# describe the dataset
train[ret_cols + volume_cols].describe()

,RET_1,RET_2,RET_3,RET_4,RET_5,RET_6,RET_7,RET_8,RET_9,RET_10,...,VOLUME_11,VOLUME_12,VOLUME_13,VOLUME_14,VOLUME_15,VOLUME_16,VOLUME_17,VOLUME_18,VOLUME_19,VOLUME_20
count,416236.000000,416130.000000,416088.000000,416051.000000,416011.000000,415998.000000,416010.000000,415972.000000,415913.000000,415903.000000,...,346570.000000,356072.000000,359587.000000,357666.000000,352222.000000,351333.000000,356281.000000,351009.000000,351266.000000,350738.000000
mean,0.001383,0.000973,0.002153,-0.000679,0.000358,-0.000261,0.000330,0.000124,-0.000621,0.000005,...,-0.084261,-0.080856,-0.075401,-0.072426,-0.085919,-0.076018,-0.087854,-0.076147,-0.076496,-0.076337
std,0.031311,0.030987,0.031332,0.031224,0.031886,0.031311,0.030966,0.031707,0.032899,0.031689,...,1.659034,2.279505,2.099580,2.317314,2.309389,2.185741,2.094459,2.423121,2.229668,2.721355
min,-0.845324,-0.770751,-0.740406,-0.863554,-0.792839,-0.723077,-0.660297,-0.806159,-0.896307,-0.753419,...,-2.184149,-2.357872,-4.228308,-5.148646,-4.388973,-5.600056,-4.610393,-4.167784,-2.341887,-2.768928
25%,-0.010970,-0.011312,-0.009769,-0.012798,-0.012249,-0.012314,-0.011416,-0.012294,-0.013011,-0.012685,...,-0.553933,-0.558539,-0.553880,-0.558473,-0.549576,-0.533574,-0.552613,-0.539491,-0.527846,-0.542790
50%,0.000637,0.000401,0.000909,-0.000495,0.000000,0.000000,0.000000,-0.000169,0.000000,0.000000,...,-0.291193,-0.291331,-0.292227,-0.293028,-0.282044,-0.272271,-0.285480,-0.281753,-0.277674,-0.283405
75%,0.012950,0.012326,0.012835,0.010813,0.011309,0.011719,0.012469,0.011481,0.011381,0.012057,...,0.040784,0.040881,0.034087,0.033964,0.026779,0.029131,0.027272,0.027639,0.030201,0.035795
max,1.444990,1.427746,3.086617,2.243749,1.491705,2.810885,1.512062,2.290456,2.471602,1.999950,...,271.947877,667.283926,453.627770,575.527471,658.209449,355.613431,408.771698,788.461460,631.249564,932.939205


In [ ]:

# Plotting the distribution of the VOLUME features
sns.boxplot(train[[f'VOLUME_{day}' for day in range(1,21)]])
plt.ylim((-1.5,1))
plt.title('VOLUME features distribution')
plt.show()

# Plotting the distribution of the RET features
sns.boxplot(train[[f'RET_{day}' for day in range(1,21)]])
plt.ylim((-1.5,1))
plt.title('RET features distribution')
plt.show()

From the boxplots it becomes obvious that a median imputation for missing values in the colums is the better choice. (The mean is too optimistic).For the returns median and mean almost coincide. For simplicity we'll choose median imputation for both.

In [ ]:
# Target balance
plt.figure(figsize=(10,5))
plt.bar(y_train['RET'].value_counts().index, y_train['RET'].value_counts().values)
plt.title('RET target distribution')
plt.show()

# Correlation matrix with return, volume and target
ret_cols = [col for col in train.columns if 'RET' in col]
vol_cols = [col for col in train.columns if 'VOLUME' in col]
corr = train[ret_cols + vol_cols].corr().abs()
plt.figure(figsize=(10,10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation matrix with return and volume features')
plt.show()


The dataset is pretty balanced. We can therefore use a conventional approach, putting inbalance-related issues aside.
About correlation analysis : there seem to be no outlier or surprising values. We can therefore assume that apart from the NaN issue, the dataset requires only a minimum cleaning. Moreover, the original features seem to be uncorrelated.

## Data Preprocessing
- Cleaning : removing all rows with no observed returns over the past 5 days
- Imputation : Simple Impute the median for the remaining NaNs of RET_x and VOLUME_x


In [14]:
# Drop all rows with too many missing returns over the past days

ret_to_drop = train[(train[ret_cols].isna().sum(axis=1)/(train[ret_cols].shape[1]) >= 0.5)][ret_cols]
train.drop(index=ret_to_drop.index, inplace=True)

In [15]:
# Simple Impute the median for the remaining NaNs of RET_x and VOLUME_x
imputer_train = SimpleImputer(strategy='median')
imputer_test = SimpleImputer(strategy='median')
impute_cols = ret_cols + volume_cols

train[impute_cols] = imputer_train.fit_transform(train[impute_cols])
test[impute_cols] = imputer_test.fit_transform(test[impute_cols])


In [16]:
missing_values = train.isna().sum().sum(), test.isna().sum().sum()  
missing_values

(0, 0)

## Feature Engineering

The main drawback in this challenge would be to deal with the noise. To do that, we could create some feature that aggregate features with some statistics. 

The following cell computes statistics on a given target conditionally to some features. For example, we want to generate a feature that describe the mean of `RET_1` conditionally to the `SECTOR` and the `DATE`.

**Ideas of improvement**: change shifts, the conditional features, the statistics, and the target. 

In [10]:
new_features = []

In [46]:

# Create new features based on the statistics by DATE aggregation for multiple shifts

shifts = [1,2,3,4,5] 
statistics = {'mean':'mean', 'std':'std','skew': lambda x: skew(x, nan_policy='omit'), 'kurt': lambda x: kurtosis(x, nan_policy='omit'), 'median':'median', 'max':'max', 'min':'min'}
target_features = ['RET','VOLUME']
for target_feature in target_features:
    tmp_name = 'DATE'
    for shift in shifts:
        for stat_name,stat in statistics.items():
            name = f'{target_feature}_{shift}_{tmp_name}_{stat_name}'
            feat = f'{target_feature}_{shift}'
            new_features.append(name)
            for data in [train, test]:
                data[name] = data.groupby(['DATE'])[feat].transform(stat)

C:\Users\Hassan\AppData\Local\Temp\ipykernel_13684\3442791746.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[name] = data.groupby(['DATE'])[feat].transform(stat)
C:\Users\Hassan\AppData\Local\Temp\ipykernel_13684\3442791746.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[name] = data.groupby(['DATE'])[feat].transform(stat)
C:\Users\Hassan\AppData\Local\Temp\ipykernel_13684\3442791746.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

In [ ]:

# Create new features based on the statistics by multiple sector and date aggregation for multiple shifts

shifts = [1,2,3,4,5] 
statistics = {'mean':'mean', 'std':'std','skew': lambda x: skew(x, nan_policy='omit'), 'kurt': lambda x: kurtosis(x, nan_policy='omit'), 'median':'median', 'max':'max', 'min':'min'}
gb_features_list = [['SECTOR', 'DATE'], ['INDUSTRY_GROUP', 'DATE'], ['INDUSTRY', 'DATE'], ['SUB_INDUSTRY', 'DATE']]
target_features = ['RET','VOLUME']
for target_feature in target_features:
    for [cat, date] in gb_features_list:
        tmp_name = cat + '_' + date
        for shift in shifts:
            for stat_name,stat in statistics.items():
                name = f'{target_feature}_{shift}_{tmp_name}_{stat_name}'
                feat = f'{target_feature}_{shift}'
                new_features.append(name)
                for data in [train, test]:
                    data[name] = data.groupby([cat, date])[feat].transform(stat)


In [ ]:
# Create new features based on rolling statistics for the past weeks

weeks = 4
target_features = ['RET', 'VOLUME'] 
for target_feature in target_features:
    for week in range(weeks):
        name = f'{target_feature}_WEEK_{week+1}'
        mean_name = 'mean_' + name
        std_name = 'std_' + name
        skew_name = 'skew_' + name
        # TODO : kurt_name = 'kurt_' + name
        new_features.extend([mean_name, std_name, skew_name])
        for data in [train, test]:
            data[mean_name] = data[[f'{target_feature}_{week*5 + day}' for day in range(1,6)]].mean(axis=1)
            data[std_name] = data[[f'{target_feature}_{week*5 + day}' for day in range(1,6)]].std(axis=1)
            data[skew_name] = data[[f'{target_feature}_{week*5 + day}' for day in range(1,6)]].skew(axis=1)
        

In [42]:
# Normalize the mean of the return and volume features by the mean of the sector and date

shifts = [1,2,3,4] 
statistics = ['sum']
gb_features_list = [['SECTOR', 'DATE']]
target_features = ['mean_VOLUME_WEEK']
for target_feature in target_features:
    for gb_features in gb_features_list:
        tmp_name = '_'.join(gb_features)
        for shift in shifts:
            for stat in statistics:
                name = f'{target_feature}_{shift}_/total_VOLUME_SECTOR_DATE'
                feat = f'{target_feature}_{shift}'
                new_features.append(name)
                for data in [train, test]:
                    data[name] = data[feat]/data.groupby(gb_features)[feat].transform('sum')

shifts = [1,2,3,4] 
statistics = ['sum'] 
gb_features_list = [['SECTOR', 'DATE']]
target_features = ['mean_RET_WEEK']
for target_feature in target_features:
    for gb_features in gb_features_list:
        tmp_name = '_'.join(gb_features)
        for shift in shifts:
            for stat in statistics:
                name = f'{target_feature}_{shift}_/total_RET_of_SECTOR_DATE'
                feat = f'{target_feature}_{shift}'
                new_features.append(name)
                for data in [train, test]:
                    data[name] = data[feat]/data.groupby(gb_features)[feat].transform('sum')


In [38]:
# Momentum features aggregation by sector and date

weeks = [1, 2, 3, 4]
targets = ['RET', 'VOLUME']

for target in targets:
    for week in weeks: 
        window_size = 5*week
        name = f'{target}_{window_size}_SECTOR_DATE_Momentum'
        new_features.append(name)
        for data in [train, test]:
            frame = data.copy()
            rolling_mean_target = frame.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(2, window_size+1)]].mean()
            target_1_mean = frame.groupby(by=['SECTOR', 'DATE'])[[f'{target}_1']].mean()
            target_1_mean_aligned, rolling_mean_target_aligned = target_1_mean.align(rolling_mean_target, axis=0, level='SECTOR')
            target_momentum = target_1_mean_aligned.sub(rolling_mean_target_aligned.mean(axis=1), axis=0)
            target_momentum.rename(columns={f'{target}_1': name},inplace=True)
            placeholder = frame.join(target_momentum, on=['SECTOR', 'DATE'], how='left')
            data[name] = placeholder[name] 

# RSI features aggregation by sector and date

targets = ["RET"]
window_size = [5, 10, 15, 20]

for window in window_size:
    name = f"RSI_{window}_SECTOR_DATE"
    new_features.append(name)
    for target in targets:
        for data in [train, test]:
            avg_gain_sector_day = data.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(1, window + 1)]].mean().agg(lambda x: x[x > 0].mean(), axis=1)
            avg_loss_sector_day = data.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(1, window + 1)]].mean().agg(lambda x: x[x < 0].mean(), axis=1).abs()
            rs_sector_day = avg_gain_sector_day / avg_loss_sector_day
            rsi_sector_date = 100 - 100 / (1 + rs_sector_day)
            data[name] = data.join(rsi_sector_date.to_frame(name), on=['SECTOR', 'DATE'], how='left')[name]


# ADL features aggregation by sector and date

window_size = [5, 10, 15, 20]
for window in window_size:
    name = f'ADL_{window}_SECTOR_DATE'
    new_features.append(name)
    for data in [train, test]:
        sum_adl = (data.groupby(by=["SECTOR", "DATE"])[[f'RET_{day}' for day in range(1, window + 1)]].apply(lambda x: (x > 0).sum()) - data.groupby(by=["SECTOR", "DATE"])[[f'RET_{day}' for day in range(1, window + 1)]].apply(lambda x: (x < 0).sum())).sum(axis=1)
        data[name] = data.join(sum_adl.to_frame(name), on=['SECTOR', 'DATE'], how='left')[name]


# Volatility features aggregation by sector and date

weeks = [1,2,3,4]
targets = ['RET', 'VOLUME']

for week in weeks: 
    window_size = 5*week
    for target in targets: 
        name = f'{target}_SECTOR_DATE_VOLATILITY_{window_size}'
        new_features.append(name)
        for data in [train, test]:
            rolling_std_target = data.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(1,window_size+1)]].mean().std(axis=1).to_frame(name)
            placeholder = data.join(rolling_std_target, on=['SECTOR', 'DATE'], how='left')
            data[name] = placeholder[name]

C:\Users\Hassan\AppData\Local\Temp\ipykernel_13684\1803907734.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[name] = placeholder[name]
C:\Users\Hassan\AppData\Local\Temp\ipykernel_13684\1803907734.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[name] = placeholder[name]
C:\Users\Hassan\AppData\Local\Temp\ipykernel_13684\1803907734.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all colum

In [39]:
# ADDITIONAL FEATURES

def compute_moving_avg(df, cols):
    return df[cols].mean(axis=1)
   
def compute_volatility(df, cols):
    return df[cols].std(axis=1)

def compute_ema(df, cols, span=5):
    return df[cols].ewm(span=span, axis=1).mean().iloc[:, -1]

def compute_momentum(df, col_start, col_end):
    return df[col_end] - df[col_start]

def compute_relative_volume(df, cols, last_col):
    return df[last_col] / df[cols].median(axis=1)

def compute_rsi(df, cols):
    gains = df[cols].clip(lower=0).mean(axis=1)
    losses = df[cols].clip(upper=0).abs().mean(axis=1)
    return 100 - (100 / (100 + (gains / losses)))

def compute_vw_ret(df, ret_cols, vol_cols):
    return (df[ret_cols] * df[vol_cols]).sum(axis=1) / df[vol_cols].sum(axis=1)


def compute_likelihood(df, ret_cols):
    positive_counts = (df[ret_cols] > 0).sum(axis=1)
    n = len(ret_cols)
    p_hat = positive_counts / n  
    likelihood = binom.cdf(k=positive_counts+1, n=n+1, p=p_hat)
    
    return likelihood

def fit_ar_n_and_predict_ret(df, n):
    # Step 1: Fit AR(n) model on RET_1 = f(RET_2, ..., RET_(n+1))
    X_train = df[[f'RET_{i}' for i in range(2, n+2)]]  # Using n lags
    y_train = df['RET_1']  # Target variable

    ar_model = LinearRegression()
    ar_model.fit(X_train, y_train)

    # Step 2: Predict RET_0 using RET_1, ..., RET_n
    X_pred = df[[f'RET_{i}' for i in range(1, n+1)]].copy()  # Copy to avoid modification warnings
    X_pred.columns = X_train.columns  # Rename to match training features to avoid error

    X_pred.columns = X_train.columns 
    df[f'RET_AR{n}_PRED'] = ar_model.predict(X_pred)
    
    return df[f'RET_AR{n}_PRED']



In [40]:

def generate_indicators(df, num_days = 20):
    ind_columns = {}

    feature_functions = {
        'MA': compute_moving_avg,
        'VOLATILITY': compute_volatility,
        'EMA': compute_ema,
        'MOMENTUM': compute_momentum,
        'REL_VOL': compute_relative_volume,
        'RSI': compute_rsi
    }
    
    for feature in ['RET', 'VOLUME']:
        cols = [f'{feature}_{i}' for i in range(1, num_days+1)]
        ind_columns[f'MA_{feature}'] = feature_functions['MA'](df, cols)
        ind_columns[f'VOLATILITY_{feature}'] = feature_functions['VOLATILITY'](df, cols)
        ind_columns[f'EMA_{feature}'] = feature_functions['EMA'](df, cols)
        ind_columns[f'MOMENTUM_{feature}'] = feature_functions['MOMENTUM'](df, f'{feature}_1', f'{feature}_{num_days}')
        if feature == 'VOLUME':
            ind_columns['REL_VOL'] = feature_functions['REL_VOL'](df, cols, f'VOLUME_{num_days}')
        else:
            ind_columns['RSI_RET'] = feature_functions['RSI'](df, cols)

    
    ind_columns['VW_RET'] = compute_vw_ret(df, [f'RET_{i}' for i in range(1, num_days+1)], [f'VOLUME_{i}' for i in range(1, num_days+1)])
    ind_columns['LIKELIHOOD_RET'] = compute_likelihood(df, [f'RET_{i}' for i in range(1, num_days+1)])
    ind_columns['RET_AR3_PRED'] = fit_ar_n_and_predict_ret(df, 3)

    for shift in [5, 10, 20]:
        ret, vol = df[[f'RET_{i}' for i in range(1, shift+1)]], df[[f'VOLUME_{i}' for i in range(1, shift+1)]]
        # returns statistics
        ind_columns[f'Mean_RET_{shift}'] = np.mean(ret, axis=1)
        ind_columns[f'Std_RET_{shift}'] = np.std(ret, axis=1)
        ind_columns[f'Range_RET_{shift}'] = (lambda x: np.max(x, axis=1) - np.min(x, axis=1))(ret)
        ind_columns[f'Momentum_RET_{shift}'] = ret.iloc[:, -1] - ret.iloc[:, 0]
        ind_columns[f'Cumulative_RET_{shift}'] = np.prod(1 + ret, axis=1) - 1
        ind_columns[f'Skew_RET_{shift}'] = skew(ret, nan_policy='omit', axis=1)
        ind_columns[f'Kurtosis_RET_{shift}'] = kurtosis(ret, nan_policy='omit', axis=1)

        # volumes statistics
        ind_columns[f'Mean_VOL_{shift}'] = np.mean(vol, axis=1)
        ind_columns[f'Std_VOL_{shift}'] = np.std(vol, axis=1)
        ind_columns[f'Skew_VOL_{shift}'] = skew(vol, nan_policy='omit', axis=1)
        ind_columns[f'Kurtosis_VOL_{shift}'] = kurtosis(vol, nan_policy='omit', axis=1)
        ind_columns[f'VOL_SURGE_{shift}'] = (df['VOLUME_1'] - ind_columns[f'Mean_VOL_{shift}']) / ind_columns[f'Std_VOL_{shift}']

        # correlation between returns and volumes
        ind_columns[f'Corr_RET_VOL_{shift}'] = [np.corrcoef(ret.iloc[i], vol.iloc[i])[0, 1] for i in range(len(ret))]

    return ind_columns

In [41]:
df_train_indicators = pd.DataFrame(generate_indicators(train))
df_test_indicators = pd.DataFrame(generate_indicators(test))

C:\Users\Hassan\AppData\Local\Temp\ipykernel_13684\269527995.py:10: FutureWarning: Support for axis=1 in DataFrame.ewm is deprecated and will be removed in a future version. Use obj.T.ewm(...) instead
  return df[cols].ewm(span=span, axis=1).mean().iloc[:, -1]
C:\Users\Hassan\AppData\Local\Temp\ipykernel_13684\269527995.py:10: FutureWarning: Support for axis=1 in DataFrame.ewm is deprecated and will be removed in a future version. Use obj.T.ewm(...) instead
  return df[cols].ewm(span=span, axis=1).mean().iloc[:, -1]
C:\Users\Hassan\AppData\Local\Temp\ipykernel_13684\269527995.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'RET_AR{n}_PRED'] = ar_model.predict(X_pred)
C:\Users\Hassan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2

In [47]:

train = pd.concat([train, df_train_indicators], axis=1)
test = pd.concat([test, df_test_indicators], axis=1)

## Data post-processing

In [ ]:
infinites_train, infinites_test = train.isin([np.inf, -np.inf]).sum().sum(), test.isin([np.inf, -np.inf]).sum().sum()
infinites_train, infinites_test # Check for infinite values

In [53]:
# infinite values
train.replace([np.inf, -np.inf], np.nan, inplace=True)
test.replace([np.inf, -np.inf], np.nan, inplace=True)

In [54]:
# NaN cols
nan_cols_train, nan_cols_test = train.isna().sum(), test.isna().sum()

In [60]:
imputer_cols_test = nan_cols_test[nan_cols_test > 0].index.tolist()
imputer_cols_train = nan_cols_train[nan_cols_train > 0].index.tolist()

In [62]:
# fill NaNs with median
imputer_train = SimpleImputer(strategy='median')
imputer_test = SimpleImputer(strategy='median')

train[imputer_cols_train] = imputer_train.fit_transform(train[imputer_cols_train])
test[imputer_cols_test] = imputer_test.fit_transform(test[imputer_cols_test])

In [63]:
train.isnull().sum().sum(), test.isnull().sum().sum()  # Checking there is no more missing values

(0, 0)

## Outputing the extended dataframe

In [64]:
# Save the new datasets
train.to_csv('../train_extended.csv')
test.to_csv('../test_extended.csv')

Next alpha factors to try :
- [https://arxiv.org/pdf/1601.00991](101 Formulaic Alphas) => https://github.com/stefan-jansen/machine-learning-for-trading/blob/main/24_alpha_factor_library/03_101_formulaic_alphas.ipynb 
- volume-return interaction features
    - VWAR
    - Ret/V
- Entropy based features
    - shannon entropy of past reurns
    - permutation entropy
    - mutual information 
- Z-score normalization
- Hurst Exponent
- Order Flow Imbalance Proxy
- Volatility Over Volume
- Price Impact Features
- Wavelet Transform feature 
    - Haar
    -Daubechies
- FFT (Fast Fourier)